# 04 — MEMO: evaluación con ejecutivo vía Ollama

Evalúa el MEMORY entrenado en el notebook 03 con el protocolo MEMO (descomposición → sub-respuestas de MEMORY → síntesis) sobre las mismas el set de evaluación (100 preguntas por defecto; EVAL_SET="12" para las 12 originales) y con el scorer copiado inline en la celda 5 (misma lógica que `memo/evaluate.py`).

**Diseño Ollama:** el **EXECUTIVE** corre vía **ChatOllama** contra el servidor local (parámetro `EXEC_OLLAMA_MODEL`: `qwen2.5:1.5b` para paridad con el run 2 en CPU, `qwen2.5:7b` para el run con modelo más grande). El **MEMORY** es el modelo entrenado (1.5B + adaptador LoRA del notebook 03) y se carga con Hugging Face en GPU — entrenar y servir un LoRA no se puede hacer vía Ollama, así que MEMORY sigue siendo HF. El EXECUTIVE nunca ve el corpus: solo las sub-respuestas de MEMORY.

**Prerrequisito:** notebook 03 (adaptador en `memo/memory_model_colab`).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
MEMORY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # base de MEMORY (misma que el entreno)
ADAPTER = REPO + "/memo/memory_model_colab"    # adaptador del notebook 03
EXEC_OLLAMA_MODEL = "qwen2.5:1.5b"   # ejecutivo: 1.5b (paridad run 2) o qwen2.5:7b (mas grande)
RUN_CONTROL = True                   # tambien evalua la base sin entrenar (control)
EVAL_SET = "100"   # set de evaluacion: "100" (default, 40/30/30) o "12" (original)
EVAL_FILE = "eval_questions_100.json" if EVAL_SET == "100" else "eval_questions.json"
SUF = "_100" if EVAL_SET == "100" else ""
N_Q = 100 if EVAL_SET == "100" else 12
GENERATE_EXPANDED = False            # True = genera set de evaluacion ampliado (solo genera; NO evalua sin revision)
EXPANDED_EVAL = REPO + "/memo/eval_questions_expanded.json"
TRACE_OUT = REPO + f"/memo/run_trace_colab{SUF}.jsonl"
TRACE_CONTROL_OUT = REPO + f"/memo/run_trace_colab_control{SUF}.jsonl"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive.
3. **Solo si no lo subes**, generarlo aquí mismo con la lógica de extracción inline (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal. La lógica es la de `pipeline/extract.py` del repo, copiada en la celda: el notebook no ejecuta scripts del repo.

`triples_seed.json`, `eval_questions.json` y `eval_questions_100.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive.")
    print("Si no lo subes, el notebook lo genera aquí mismo con la lógica inline (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar los chunks aquí mismo...")
        print("(lógica de extracción inline: descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        # El PDF está cifrado con AES (contraseña vacía): pypdf necesita el
        # paquete opcional `cryptography` para leerlo, que `pip install pypdf`
        # no trae. Se verifica aquí para que el último recurso funcione.
        try:
            import pypdf
            import cryptography
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1", "cryptography>=3.1"], check=True)
            import pypdf
            import cryptography
        import urllib.request
        from pypdf import PdfReader

        # Lógica de extracción del informe, copiada inline: este notebook no
        # ejecuta scripts del repo. Solo la edición en inglés: el sitio en
        # español bloquea las descargas directas (WAF), así que la edición en
        # inglés es la que se descarga (mismo informe, mismas cifras).
        _PDF_URL_EN = "https://annualreport.femsa.com/pdf/FEMSA_2024_Annual_Report.pdf"
        _PDF_PATH_EN = Path(DATA_DIR) / "FEMSA_2024_Annual_Report_EN.pdf"
        _SOURCE_LABEL_EN = "FEMSA 2024 Integrated Annual Report, «Evolving» (English edition, same figures)"

        def _inline_is_valid_pdf(path):
            try:
                return len(PdfReader(str(path)).pages) > 0
            except Exception:
                return False

        def _inline_download_pdf():
            if _PDF_PATH_EN.exists() and _inline_is_valid_pdf(_PDF_PATH_EN):
                print(f"PDF en caché: {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
                return _PDF_PATH_EN
            if _PDF_PATH_EN.exists():
                print(f"Caché inválido, se vuelve a descargar: {_PDF_PATH_EN}")
                _PDF_PATH_EN.unlink()
            Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Descargando {_PDF_URL_EN} ...")
            req = urllib.request.Request(
                _PDF_URL_EN, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as resp:
                with open(_PDF_PATH_EN, "wb") as fh:
                    fh.write(resp.read())
            if not _inline_is_valid_pdf(_PDF_PATH_EN):
                _PDF_PATH_EN.unlink()
                raise RuntimeError("El archivo descargado no es un PDF válido.")
            print(f"Guardado en {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
            return _PDF_PATH_EN

        def _inline_extract_pages(pdf_path):
            reader = PdfReader(str(pdf_path))
            chunks = []
            for i, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").strip()
                chunks.append({"page": i, "text": text, "chars": len(text),
                               "fuente": _SOURCE_LABEL_EN})
            return chunks

        try:
            _pdf = _inline_download_pdf()
            _chunks = _inline_extract_pages(_pdf)
            _non_empty = sum(1 for c in _chunks if c["chars"] > 0)
            print(f"Páginas: {len(_chunks)} ({_non_empty} con texto)")
            Path(CHUNKS).parent.mkdir(parents=True, exist_ok=True)
            with open(CHUNKS, "w", encoding="utf-8") as fh:
                for c in _chunks:
                    fh.write(json.dumps(c, ensure_ascii=False) + "\n")
            print(f"{len(_chunks)} páginas -> {CHUNKS}")
        except Exception as e:
            print("Generación inline falló:", e)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 accelerate>=1.16.0 peft>=0.22.0 numpy>=2.5.5 langchain-ollama

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=EXEC_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", EXEC_OLLAMA_MODEL)

## Prerrequisito: el adaptador MEMORY del notebook 03

Sin el adaptador entrenado no hay nada que evaluar. Esta celda falla con un mensaje claro si no existe: corre el notebook 03 primero.

In [ ]:
import os
assert os.path.isdir(ADAPTER), (
    "FALTA el adaptador MEMORY: " + ADAPTER + "\n"
    "Lo produce el notebook 03 (MEMO entrenamiento). Corre 03 primero; "
    "sin adaptador entrenado, la evaluación no tiene sentido."
)
print("adaptador MEMORY OK:", ADAPTER)

## 1. Cargar MEMORY (HF, GPU) y EXECUTIVE (Ollama)

MEMORY = base 1.5B + adaptador LoRA del notebook 03 (o la base sola si `RUN_CONTROL`). EXECUTIVE = ChatOllama contra el servidor local. Los prompts y la descomposición por reglas están copiados inline en la celda siguiente (origen: `memo/executive_v2.py`); el notebook no importa código del repo.

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

exec_llm = ChatOllama(model=EXEC_OLLAMA_MODEL, base_url="http://localhost:11434",
                      temperature=0, num_predict=150)
print("EXECUTIVE (Ollama):", EXEC_OLLAMA_MODEL)

# Prompts y descomposicion por reglas, copiados inline (misma logica y mismo
# texto que en el script del repo; el notebook es autocontenido).
DECOMP_SYSTEM = (
    "Descompón la pregunta del usuario en 1 a 3 sub-preguntas simples, cada una "
    "sobre un solo hecho del informe anual de FEMSA 2024. Responde solo con las "
    "sub-preguntas, una por línea, sin numerar. Crea sub-preguntas nuevas sobre "
    "la pregunta del usuario; NO copies las preguntas de los ejemplos.")
DECOMP_EXAMPLES = (
    "Pregunta: ¿Qué división aportó más ingresos en 2024, Proximidad Américas o Coca-Cola FEMSA, y por cuánto?\n"
    "¿Cuáles fueron los ingresos de Proximidad Américas en 2024?\n"
    "¿Cuáles fueron los ingresos de Coca-Cola FEMSA en 2024?\n\n"
    "Pregunta: ¿Qué porcentaje de los ingresos consolidados 2024 representan Proximidad Américas y Coca-Cola FEMSA juntas?\n"
    "¿Cuáles fueron los ingresos totales consolidados de FEMSA en 2024?\n"
    "¿Cuáles fueron los ingresos de Proximidad Américas en 2024?\n"
    "¿Cuáles fueron los ingresos de Coca-Cola FEMSA en 2024?\n\n")


def rule_decompose(q):
    '''Descomposicion por reglas (fallback documentado).'''
    low = q.lower()
    divs = ["Proximidad Américas", "Proximidad Europa", "Coca-Cola FEMSA",
            "Salud", "Combustibles", "Spin"]
    named = [d for d in divs if d.lower() in low]
    if len(named) >= 2 and ("ingresos" in low or "creci" in low or "aport" in low):
        metric = "ingresos" if "ingresos" in low else "crecimiento"
        return [f"¿Cuáles fueron los {metric} de {d} en 2024?" for d in named]
    if "porcentaje" in low and "juntas" in low:
        return ["¿Cuáles fueron los ingresos totales consolidados de FEMSA en 2024?",
                "¿Cuáles fueron los ingresos de Proximidad Américas en 2024?",
                "¿Cuáles fueron los ingresos de Coca-Cola FEMSA en 2024?"]
    if "creció más" in low or "crecio mas" in low:
        return [q]  # la comparacion la resuelve el EXECUTIVE con las dos cifras
    return [q]


def chat_ollama(system, user):
    msgs = [SystemMessage(content=system), HumanMessage(content=user)]
    return exec_llm.invoke(msgs).content.strip()


mem_tok = AutoTokenizer.from_pretrained(MEMORY_MODEL)
print(f"cargando MEMORY {MEMORY_MODEL} (bf16, device_map=auto)...", flush=True)
_mem_base = AutoModelForCausalLM.from_pretrained(MEMORY_MODEL,
                                                 torch_dtype=torch.bfloat16,
                                                 device_map="auto")
print(f"cargando adaptador {ADAPTER} ...", flush=True)
mem_model = PeftModel.from_pretrained(_mem_base, ADAPTER)
mem_model.eval()
print("MEMORY lista: base + adaptador")

## 2. Protocolo MEMO (lógica inline, origen: `memo/executive_v2.py`)

Mismo protocolo que el run 2: descomposición por LLM (ahora vía Ollama; si falla el parseo, cae a la regla documentada), sub-respuestas de MEMORY (greedy, cortas) y síntesis por el EXECUTIVE. La traza guarda el método de descomposición (`llm` o `rule`) por pregunta, como en el original.

In [ ]:
def llm_decompose(q):
    out = chat_ollama(DECOMP_SYSTEM, DECOMP_EXAMPLES + f"Pregunta: {q}\n")
    lines = [l.strip().lstrip("0123456789.-) ").strip()
             for l in out.splitlines() if l.strip().endswith("?")]
    lines = [l for l in lines if len(l) > 10][:3]
    return lines or None


def memory_answer(subq):
    system = ("Responde con el dato exacto del informe anual de FEMSA 2024, "
              "en una frase corta. Si no sabes el dato, di 'No lo sé'.")
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": subq}]
    text = mem_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = mem_tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = mem_model.generate(**ids, max_new_tokens=60, do_sample=False,
                                 pad_token_id=mem_tok.eos_token_id)
    return mem_tok.decode(out[0][ids["input_ids"].shape[1]:],
                          skip_special_tokens=True).strip()


def synthesize(q, sub_answers):
    context = "\n".join(f"- {s}: {a}" for s, a in sub_answers)
    system = ("Responde la pregunta usando SOLO la información siguiente, "
              "que viene del modelo de memoria. No inventes cifras. "
              "Si la información no alcanza, dilo.")
    user = f"Información del modelo de memoria:\n{context}\n\nPregunta: {q}"
    return chat_ollama(system, user)


def run_memo(q):
    subs = llm_decompose(q)
    method = "llm"
    if not subs:
        subs = rule_decompose(q)
        method = "rule"
    sub_answers = [(s, memory_answer(s)) for s in subs]
    final = synthesize(q, sub_answers)
    return {"question": q, "decomposition": method,
            "exec_model": f"ollama {EXEC_OLLAMA_MODEL}",
            "memory_model": MEMORY_MODEL,
            "sub_questions": subs,
            "sub_answers": [{"sub_question": s, "memory_answer": a}
                            for s, a in sub_answers],
            "final_answer": final}

## 3. Correr el set de evaluación (MEMORY entrenado)

In [ ]:
import json

eval_qs = json.load(open(REPO + "/memo/" + EVAL_FILE))["questions"]
with open(TRACE_OUT, "w") as f:
    for item in eval_qs:
        r = run_memo(item["question"])
        r["id"] = item["id"]
        r["level"] = item["level"]
        r["control"] = False
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"{item['id']} [{r['decomposition']}] -> {r['final_answer'][:120]}", flush=True)
print(f"traza -> {TRACE_OUT}")

## 4. Control (base sin entrenar, mismo ejecutivo)

Si `RUN_CONTROL` es True, repite el set con MEMORY = base sin adaptador. El control mide cuánto viene del entrenamiento y cuánto del ejecutivo: en el run 2 el control sacó 0/12 con el mismo ejecutivo 1.5B.

In [ ]:
if RUN_CONTROL:
    print("cargando MEMORY control (base sin adaptador)...", flush=True)
    _base = AutoModelForCausalLM.from_pretrained(MEMORY_MODEL,
                                                 torch_dtype=torch.bfloat16,
                                                 device_map="auto")
    _base.eval()
    mem_model = _base
    eval_qs = json.load(open(REPO + "/memo/" + EVAL_FILE))["questions"]
    with open(TRACE_CONTROL_OUT, "w") as f:
        for item in eval_qs:
            r = run_memo(item["question"])
            r["id"] = item["id"]
            r["level"] = item["level"]
            r["control"] = True
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            print(f"{item['id']} [{r['decomposition']}] -> {r['final_answer'][:120]}", flush=True)
    print(f"traza control -> {TRACE_CONTROL_OUT}")
else:
    print("RUN_CONTROL=False: control omitido.")

## 5. Puntuar con el scorer inline

El scorer está copiado inline en la celda siguiente (origen: `memo/evaluate.py`): misma exactitud numérica con tolerancia de redondeo, misma regla de alucinación (cualquier número en la respuesta que no esté verificado en el informe), misma regla Q3 con 1,596 y mismas keywords para las preguntas sin números esperados (Q8/Q10/Q11). Escribe los mismos archivos (`results_colab{SUF}.md` y `results_colab_control{SUF}.md`). Compara con el run 2 en CPU (2/12) y el run 1 (0/12).

In [ ]:
import json, re


def digits(s):
    return re.sub(r"[.,]", "", s)


def numbers_in(text):
    return {digits(m.group(0)) for m in re.finditer(r"\d[\d.,]*", text or "")}


# Normaliza un numero esperado ('781585', '11.2') a digitos comparables.
def norm_num(x):
    return digits(str(x))


# True si cada numero esperado aparece (o su redondeo a entero) en la respuesta.
def expected_hit(answer_nums, expected):
    for e in expected:
        d = norm_num(e)
        if d in answer_nums:
            continue
        # tolerancia de redondeo: 307197 vs 307200, 279793 vs 279800
        try:
            v = float(e)
            ok = any(abs(float(a) - v) <= max(1.0, abs(v) * 0.001)
                     for a in answer_nums if a.replace(".", "", 1).isdigit())
        except ValueError:
            ok = False
        if not ok:
            return False
    return True


def score_trace(trace, out):
    # Scorer inline (misma logica que el scorer del repo): exactitud numerica
    # con tolerancia de redondeo, alucinacion = numero no verificado en el
    # informe, regla Q3 con 1,596 y keywords para preguntas sin numeros.
    eval_data = json.load(open(REPO + "/memo/" + EVAL_FILE))
    questions = {q["id"]: q for q in eval_data["questions"]}
    trace_rows = [json.loads(l) for l in open(trace)]

    # conjunto de numeros verificados del informe
    verified = set()
    try:
        for line in open(REPO + "/memo/reflection_qa_v3.jsonl"):
            r = json.loads(line)
            verified |= numbers_in(r["answer"])
    except FileNotFoundError:
        pass
    try:
        seed = json.load(open(REPO + "/pipeline/triples_seed.json"))
        triples = seed if isinstance(seed, list) else seed.get("triples", [])
        for t in triples:
            for k in ("valor", "variacion_pct"):
                if t.get(k) is not None:
                    verified.add(norm_num(t[k]))
    except FileNotFoundError:
        pass
    # anios y periodos no son alucinaciones
    verified |= {"2024", "2023", "2022"}
    # las cifras esperadas del propio set de evaluacion tambien estan
    # verificadas (con 100 preguntas, si no, respuestas correctas nuevas se
    # marcarian como alucinacion por no estar en reflection_qa/triples_seed)
    for _q in questions.values():
        verified |= numbers_in(_q.get("expected", ""))
        verified |= {norm_num(n) for n in _q.get("expected_numbers", [])}

    rows = []
    for r in trace_rows:
        q = questions[r["id"]]
        ans = r["final_answer"]
        ans_nums = numbers_in(ans)
        halluc = sorted(n for n in ans_nums if n not in verified and len(n) > 1)
        ok = expected_hit(ans_nums, q.get("expected_numbers", []))
        # Preguntas sin numeros esperados: se evaluan por palabras clave.
        # Sin keywords ni numeros, NUNCA se marca correcta automaticamente
        # (el bug original: expected_hit([]) devolvia True por vacuidad y
        # Q8/Q10/Q11 salian "correctas" con respuestas erroneas).
        kws = q.get("expected_keywords", [])
        if not q.get("expected_numbers") and kws:
            low = ans.lower()
            ok = all(k.lower() in low for k in kws)
        elif not q.get("expected_numbers") and not kws:
            ok = False
        note = ""
        if q.get("ground_truth_note"):
            note = "Nota ground-truth: " + q["ground_truth_note"]
        if r["id"] == "Q3" and "1596" in ans_nums:
            ok = True
            note += " (El modelo respondio 1,596 = valor real del informe; se cuenta como correcto.)"
        verdict = "correcta" if ok and not halluc else (
            "correcta con alucinacion" if ok else "incorrecta")
        rows.append({"id": r["id"], "level": q["level"], "question": q["question"],
                     "expected": q["expected"], "answer": ans,
                     "decomposition": r.get("decomposition"),
                     "verdict": verdict, "hallucinated": halluc, "note": note})

    by_level = {}
    for row in rows:
        by_level.setdefault(row["level"], []).append(row)
    total_ok = sum(1 for r in rows if r["verdict"] == "correcta")
    total_halluc = sum(1 for r in rows if r["hallucinated"])

    lines = ["# Resultados MEMO (demo) — FEMSA 2024",
             "",
             "Sistema: EXECUTIVE = Qwen2.5-0.5B-Instruct base (congelado); "
             "MEMORY = base + adaptador LoRA entrenado en reflexiones del informe. "
             "El EXECUTIVE nunca ve el corpus; solo las sub-respuestas de MEMORY.",
             "",
             f"Correctas: **{total_ok}/{len(rows)}**. Con alucinacion (numero no verificado en el informe): **{total_halluc}/{len(rows)}**.",
             "",
             "| ID | Nivel | Pregunta | Esperado | Respuesta MEMO | Veredicto |",
             "|----|-------|----------|----------|----------------|-----------|"]
    for r in rows:
        a = r["answer"].replace("|", "/").replace("\n", " ")[:220]
        lines.append(f"| {r['id']} | {r['level']} | {r['question'][:70]} | "
                     f"{r['expected'][:60]} | {a} | {r['verdict']} |")
    lines += ["",
              "## Por nivel",
              ""]
    for lv in sorted(by_level):
        ok = sum(1 for r in by_level[lv] if r["verdict"] == "correcta")
        lines.append(f"- Nivel {lv}: {ok}/{len(by_level[lv])} correctas.")
    lines += ["",
              "## Comparacion con el grafo (declarada, no medida con numeros inventados)",
              "",
              "- Grafo (Cypher sobre tripletas semilla): ~perfecto en nivel 1-2 **por "
              "construccion**, porque las respuestas son las tripletas verificadas. "
              "En nivel 3 depende de que el agente sepa agregar sin inventar.",
              "- MEMO (demo): ver tabla. La memoria es un modelo entrenado, no una base "
              "de datos: puede equivocar cifras aunque haya visto el hecho en "
              "entrenamiento. Esa es justo la diferencia que el post debe mostrar.",
              "- RAG vectorial: **pendiente** (no se corrio en este demo; no se "
              "inventan cifras).",
              "",
              "## Notas de ground-truth",
              ""]
    for r in rows:
        if r["note"]:
            lines.append(f"- {r['id']}: {r['note']}")
    if not any(r["note"] for r in rows):
        lines.append("- Ninguna.")
    lines += ["",
              "## Alucinaciones detectadas",
              ""]
    for r in rows:
        if r["hallucinated"]:
            lines.append(f"- {r['id']}: numeros no verificados en el informe: "
                         f"{', '.join(r['hallucinated'])}")
    if not any(r["hallucinated"] for r in rows):
        lines.append("- Ninguna.")

    open(out, "w").write("\n".join(lines) + "\n")
    print(f"correctas {total_ok}/{len(rows)}, con alucinacion {total_halluc}/{len(rows)} -> {out}")
    for r in rows:
        print(f"  {r['id']} L{r['level']} [{r['decomposition']}] {r['verdict']}"
              + (f" halluc={r['hallucinated']}" if r["hallucinated"] else ""))


for trace, out in [(TRACE_OUT, REPO + f"/memo/results_colab{SUF}.md"),
                   (TRACE_CONTROL_OUT, REPO + f"/memo/results_colab_control{SUF}.md")]:
    if RUN_CONTROL or "control" not in trace:
        score_trace(trace, out)

## 6. (Opcional) Generar un set de evaluación ampliado

Con `GENERATE_EXPANDED = True` esta celda genera preguntas nuevas a partir de las tripletas semilla (hechos ya verificados) con plantillas por tipo de tripleta. **Cada cifra esperada se verifica por grep literal en el corpus (`chunks.jsonl`): lo que no aparece literal, se descarta — nunca se adivina.** También se descartan las que se parecen demasiado (Jaccard ≥ 0.80) a las preguntas del set actual (EVAL_FILE) originales o entre sí. La salida lleva `"needs_review": true`: hay que revisar las preguntas a mano antes de evaluar con ellas (cambia `EVAL_FILE` en la celda 3 y vuelve a correr). Por defecto está apagado.

In [ ]:
import json, re

if not GENERATE_EXPANDED:
    print("GENERATE_EXPANDED=False: no se genera nada. Ponlo en True para generar el set.")
else:
    triples = json.load(open(REPO + "/pipeline/triples_seed.json"))["triples"]
    chunks = [json.loads(l) for l in open(REPO + "/pipeline/data/chunks.jsonl")]
    corpus = "\n".join(c["text"] for c in chunks)
    orig = json.load(open(REPO + "/memo/" + EVAL_FILE))["questions"]

    def tokens(q):
        import unicodedata
        q = unicodedata.normalize("NFKD", q.lower())
        q = "".join(c for c in q if not unicodedata.combining(c))
        return set(re.findall(r"[a-z0-9]+", q))

    def jaccard(a, b):
        return len(a & b) / len(a | b) if a | b else 0.0

    def literal_present(text, expected):
        e = str(expected).replace(",", "")
        if "." in e:
            return e in text or e.replace(".", ",") in text
        try:
            grouped = f"{int(e):,}"
        except ValueError:
            return e in text
        return grouped in text or e in text

    orig_toks = [tokens(q["question"]) for q in orig]
    cands = []
    for t in triples:
        subj, rel, obj = t.get("subject"), t.get("relation"), t.get("object")
        valor, var = t.get("valor"), t.get("variacion_pct")
        if valor is None:
            continue
        if rel == "reporta" and obj == "Ingresos Totales":
            q = f"¿Cuáles fueron los ingresos totales de {subj} en 2024?"
            exp_nums = [str(valor)] + ([str(var)] if var is not None else [])
            exp = f"{valor} (Ps. millones)" + (f", +{var}%" if var is not None else "")
        elif rel == "mide" and obj == "Usuarios Activos":
            q = f"¿Cuántos usuarios activos tuvo {subj} en 2024?"
            exp_nums = [str(valor)] + ([str(var)] if var is not None else [])
            exp = f"{valor} millones" + (f" (+{var}%)" if var is not None else "")
        elif rel == "mide" and obj == "Tiendas":
            q = f"¿Cuántas tiendas tenía {subj} al cierre de 2024?"
            exp_nums = [str(valor)]
            exp = f"{valor} tiendas"
        elif rel == "mide" and obj == "Estaciones":
            q = f"¿Cuántas estaciones tenía OXXO GAS al cierre de 2024?"
            exp_nums = [str(valor)]
            exp = f"{valor} estaciones"
        else:
            continue
        cands.append({"question": q, "expected": exp, "expected_numbers": exp_nums,
                      "source_triple": {k: t.get(k) for k in ("subject", "relation", "object", "valor", "variacion_pct")}})

    kept, dropped = [], []
    seen_q = set()
    for c in cands:
        key = tokens(c["question"])
        if c["question"] in seen_q:
            dropped.append((c["question"], "duplicada"))
            continue
        if max((jaccard(key, et) for et in orig_toks), default=0) >= 0.80:
            dropped.append((c["question"], "demasiado parecida a una de las 12"))
            continue
        missing = [e for e in c["expected_numbers"] if not literal_present(corpus, e)]
        if missing:
            dropped.append((c["question"], f"cifra no verificada por grep literal: {missing}"))
            continue
        seen_q.add(c["question"])
        kept.append(c)

    out = {"needs_review": True,
           "note": "Generado por plantillas desde triples_seed.json; cada cifra verificada por grep literal en chunks.jsonl. REVISAR A MANO antes de evaluar.",
           "questions": [{"id": f"X{i+1}", "level": 1, **c} for i, c in enumerate(kept)]}
    json.dump(out, open(EXPANDED_EVAL, "w"), ensure_ascii=False, indent=2)
    print(f"candidatas: {len(cands)} | aceptadas: {len(kept)} | descartadas: {len(dropped)}")
    for q, why in dropped:
        print(f"  DESCARTADA: {q} ({why})")
    for c in kept:
        print(f"  X: {c['question']} -> {c['expected']}")
    print(f"-> {EXPANDED_EVAL} (needs_review=true: revisa antes de usar)")

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('memo/run_trace_colab{SUF}.jsonl', 'Traza MEMO Colab ({N_Q} preguntas)', True), ('memo/run_trace_colab_control{SUF}.jsonl', 'Traza control (base sin entrenar)', False), ('memo/results_colab{SUF}.md', 'Resultados MEMO Colab', True), ('memo/results_colab_control{SUF}.md', 'Resultados control', False), ('memo/eval_questions_expanded.json', 'Set de evaluación ampliado (si se generó; needs_review=true)', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    # {SUF}/{N_Q} se resuelven con los parametros del notebook (EVAL_SET);
    # en notebooks sin esos parametros valen "" y 12.
    rel = rel.format(SUF=globals().get("SUF", ""), N_Q=globals().get("N_Q", 12))
    label = label.format(SUF=globals().get("SUF", ""), N_Q=globals().get("N_Q", 12))
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)